## AISE4010- Assignment3 - Time Series Classification using TCN and Transformer + Hyperparameter Tuning

## Grade: 100 points

### Instructions

#### Follow These Steps before submitting your assignment

1. Complete the notebook.

2. Make sure all plots have axis labels.

3. Once the notebook is complete, `Restart` your kernel by clicking 'Kernel' > 'Restart & Run All'.

4. Fix any errors until your notebook runs without any problems.

5. Submit one completed notebook for the group to OWL by the deadline.

6. Make sure to reference all external code and documentation used.

### Dataset

The dataset is a sample of 46 satellite images, collected in 2006, located in southwestern France near Toulouse. It
is a 24 km × 24 km area and the dataset uses 3 output classes (2 available) for arable soil classification based on the following paper: https://arxiv.org/pdf/1811.10166.

You will be using helper functions below to prepare it for deep learning models.

In [2]:
# Call this helper method by passing in the names of the provided training and test sets' files.
def read_SITS_data(name_file):
    data = pd.read_table(name_file, sep=',', header=None)

    y_data = data.iloc[:,0]
    y = np.asarray(y_data.values, dtype='uint8')
    y[y>1] = 0

    polygonID_data = data.iloc[:,1]
    polygon_ids = polygonID_data.values
    polygon_ids = np.asarray(polygon_ids, dtype='uint16')

    X_data = data.iloc[:,2:]
    X = X_data.values
    X = np.asarray(X, dtype='float32')

    return  X, polygon_ids, y

In [3]:
def custom_feature_scaling(train, test):
    min_per = np.percentile(train, 2, axis=(0,1))
    max_per = np.percentile(train, 100-2, axis=(0,1))

    new_train = (train-min_per)/(max_per-min_per)
    new_test = (test-min_per)/(max_per-min_per)

    return new_train, new_test

### Question 1 - Data Preprocessing (15%)
- Q1.1 Call "read_SITS_data()" for the training set and store the results as X_train, polygon_ids_train, and y_train.
- Q1.2 Call "read_SITS_data()" above for the test set and store the results as X_test, polygon_ids_test, and y_test.
- Q1.3 Reshape the training and test sets.
  - Each set must be reshaped into a 3-D array. The first dimension will be the number of rows of the original set. The second dimension will be int(x / 3), where x is the number of columns of the original set and int() is a casting function. The third dimension will be 3 (number of channels).
- Q1.4 Call "custom_feature_scaling()" with the training and test sets. Save the results as the final sets for use.
- Q1.5 How many entries are in the training set? How many time steps are in each entry? How many features are there for each time step? How many labels for each entry?


In [15]:
import pandas as pd
import numpy as np

#read training set to load datasets and store results 
X_train, polygon_ids_train, y_train = read_SITS_data("C:/Users/sixel/fourth_year/4410/assignments/3/train_dataset.csv")

#read testing set to load datasets and store results 
X_test, polygon_ids_test, y_test = read_SITS_data("C:/Users/sixel/fourth_year/4410/assignments/3/test_dataset.csv")

# Compute number of time steps
timesteps_train = X_train.shape[1] // 3
timesteps_test = X_test.shape[1] // 3

#reshape the train/test sets to (samples, time_steps, 3)
X_train = X_train.reshape(X_train.shape[0], timesteps_train, 3)
X_test  = X_test.reshape(X_test.shape[0], timesteps_test, 3)

#call function for custom scaling and store results
X_train_final, X_test_final = custom_feature_scaling(X_train, X_test)

#summary
print("Training entries:", X_train_final.shape[0])
print("Time steps per entry:", X_train_final.shape[1])
print("Features per time step:", X_train_final.shape[2])
print("Labels per entry:", y_train.shape[0])

Training entries: 260
Time steps per entry: 149
Features per time step: 3
Labels per entry: 260


*Write your Answer to Q1.5 here:* There are 260 training entries, 149 time steps, 3 features per time step and 260 labels per entry.



### Question2 - Temporal Convolutional Network
- Q2.1 Create a Sequential model for classification. The model should have a TCN layer of size 64, a fully connected layer of size 256, a dropout of 0.3, and a fully connected output layer with Softmax activation (Hint: the logits axis should be on 0). Train the model using the provided dataset for 20 epochs. Use the batch_size of 32, and ADAM optimizer. Print the model summary.
- Q2.2 Train the model with the same parameters, print the model summary and evaluate the model's accuracy on the test set. Print the accuracy.
- Q2.3 Why do we use the Softmax activation on the output layer? In what scenarios does this contrast to using ReLU instead?


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tcn import TCN

#classification means dataset has labels 0 or 1
num_classes = 2  


#build model
model = Sequential([
    #TCN layer of 64 neurons
    TCN(nb_filters=64, input_shape=(X_train_final.shape[1], 3)),
    #fully connected layer with 256 neurons
    Dense(256, activation='relu'),
    #dropout layer
    Dropout(0.3),
    #fully connected layer with softmax
    Dense(num_classes, activation='softmax')
])
#compile the model
model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


#print summary for visual
model.summary()
#train model
history = model.fit(
    X_train_final, 
    y_train, 
    batch_size=32,
    epochs=20,
    validation_split=0.1,
    verbose=1)
#evaluate the model
test_loss, test_acc = model.evaluate(X_test_final, y_test)
print("Test Accuracy:", test_acc)

c:\Users\sixel\AppData\Local\Programs\Python\Python312\Lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tcn (TCN)                       │ (None, 64)             │       136,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 153,922 (601.26 KB)

 Trainable params: 153,922 (601.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 88ms/step - accuracy: 0.8103 - loss: 1.2310 - val_accuracy: 1.0000 - val_loss: 0.0931
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.9036 - loss: 0.2711 - val_accuracy: 1.0000 - val_loss: 0.0813
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8523 - loss: 0.2775 - val_accuracy: 1.0000 - val_loss: 0.0102
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9156 - loss: 0.2072 - val_accuracy: 1.0000 - val_loss: 0.0752
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9309 - loss: 0.1396 - val_accuracy: 1.0000 - val_loss: 0.0217
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9202 - loss: 0.1501 - val_accuracy: 1.0000 - val_loss: 0.0105
Epoch 7/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9601 - loss: 0.1032 - val_accuracy: 1.0000 - val_loss: 0.0855
Epoch 8/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9683 - loss: 0.0849 - val_accuracy: 1.0000 - val_loss: 0.0085


*Write your Answer to Q2.3 Here:* Softmax is used because this is a classification task. It converts the output into probabilities that all sum to 1, telling the model how likely class 1/0 is allowing the choice with the highest probability to be chosen. Others like ReLU cannot be chosen because its outputs do not sum to 1, cannot be interpreted as probabilities and can output 0 for multiple classes. ReLU would be better used for regression tasks.




### Question 3 - Transformer Model
- Q3.1 Create a transformer encoder block. It should use MultiHeadAttention for residual connection. The projection layers can be two Conv1D layers, based on number of feed forward dimensions and with kernel sizes of 1.
- Q3.2 Define the model. It should have 4 encoder blocks, each with 256 heads and feed forward dimensions of 4. Add a flatten layer, then a fully connected layer of size 2 and a fully connected output layer.
- Q3.3 Print the model summary, train the model using 50 epochs and a batch size of 32. Evaluate the model accuracy on the test set and print it.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, MultiHeadAttention, LayerNormalization, Conv1D, Dropout, Add
from tensorflow.keras import Sequential

#transformer encoder block
def transformer_encoder_block(inputs, num_heads, ff_dim, dropout=0.1):
    #multi-head attention
    attn_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=inputs.shape[-1],
        dropout=dropout
    )(inputs, inputs)
    #residual connection
    out1 = Add()([inputs, attn_output])
    out1 = LayerNormalization(epsilon=1e-6)(out1)
    #feed-forward network using Conv1D
    ff_output = Sequential([
        Conv1D(filters=ff_dim, kernel_size=1, activation='relu'), #kernel size 1
        Conv1D(filters=inputs.shape[-1], kernel_size=1) 
    ])(out1)
    # Residual connection 2
    out2 = Add()([out1, ff_output])
    out2 = LayerNormalization(epsilon=1e-6)(out2)

    return out2


from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model
num_heads = 256 #attention heads
ff_dim = 4 #feed forward dimension in each block
num_classes = 2 #two output classes for classification
inputs = Input(shape=(X_train_final.shape[1], X_train_final.shape[2])) #input shape

#add 4 transformer encoder blocks in sequence
x = inputs
for _ in range(4):
    x = transformer_encoder_block(x, num_heads=num_heads, ff_dim=ff_dim)

x = Flatten()(x) #make 2d vector
x = Dense(2, activation='relu')(x) #fully connected layer of size 2
outputs = Dense(num_classes, activation='softmax')(x) #output layer with softmax 

transformer_model = Model(inputs, outputs) #build model

transformer_model.compile( 
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

transformer_model.summary() #print model summary


#train using 50 epochs
history = transformer_model.fit(
    X_train_final,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)
test_loss, test_acc = transformer_model.evaluate(X_test_final, y_test)
print("Transformer Test Accuracy:", test_acc)

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 149, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 149, 3)    │     11,523 │ input_layer_6[0]… │
│ (MultiHeadAttentio… │                   │            │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 149, 3)    │          0 │ input_layer_6[0]… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 149, 3)    │          6 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_5        │ (None, 149, 3)    │         31 │ layer_normalizat… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 149, 3)    │          0 │ layer_normalizat… │
│                     │                   │            │ sequential_5[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 149, 3)    │          6 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 149, 3)    │     11,523 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 149, 3)    │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 149, 3)    │          6 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_6        │ (None, 149, 3)    │         31 │ layer_normalizat… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 149, 3)    │          0 │ layer_normalizat… │
│                     │                   │            │ sequential_6[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 149, 3)    │          6 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 149, 3)    │     11,523 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 149, 3)    │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 149, 3)    │          6 │ add_12[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_7        │ (None, 149, 3)    │         31 │ layer_normalizat

 Total params: 47,166 (184.24 KB)

 Trainable params: 47,166 (184.24 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - accuracy: 0.8480 - loss: 0.7310 - val_accuracy: 1.0000 - val_loss: 0.6853
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.9159 - loss: 0.6852 - val_accuracy: 1.0000 - val_loss: 0.6776
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - accuracy: 0.9243 - loss: 0.6785 - val_accuracy: 1.0000 - val_loss: 0.6700
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - accuracy: 0.9157 - loss: 0.6724 - val_accuracy: 1.0000 - val_loss: 0.6624
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - accuracy: 0.9089 - loss: 0.6666 - val_accuracy: 1.0000 - val_loss: 0.6548
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - accuracy: 0.9044 - loss: 0.6608 - val_accuracy: 1.0000 - val_loss: 0.6473
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - accuracy: 0.9000 - loss: 0.6552 - val_accuracy: 1.0000 - val_loss: 0.6398
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - accuracy: 0.8941 - loss: 0.6500 - val_accuracy: 1.0000 - val_loss: 0.6326
Epoch 9/

### Question 4 - Hyperparameter Tuning
- Q4.1 Define a search space for the number of neurons in the fully connected layer that follows the flatten layer. The lower bound should be 2, the upper bound should be 16, and it should search every other value in between. Also have the tuner decide whether or not a dropout layer of 0.3 should be added after the aforementioned layer.
- Q4.2 Using GridSearch, search for the best hyperparameters with respect to accuracy over 50 epochs.
- Q4.3 Using the best hyperparameters, rebuild the model and print the model accuracy.

In [ ]:
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import GridSearchCV
import numpy as np

#build model
def build_model(fc_units=2, use_dropout=False):
    model = Sequential()
    model.add(Flatten(input_shape=(X_train_final.shape[1], X_train_final.shape[2])))

    #fully connected layer
    model.add(Dense(fc_units, activation='relu'))
    #optional dropout
    if use_dropout:
        model.add(Dropout(0.3))
    #output layer
    model.add(Dense(2, activation='softmax'))
    #compile model
    model.compile(
        optimizer=Adam(),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

#hyperparameter search space
param_grid = {
    "model__fc_units": np.arange(2, 17, 2),
    "model__use_dropout": [True, False],
    "epochs": [50],
    "batch_size": [32]
}


clf = KerasClassifier(model=build_model, verbose=0) #wrap keras model so grid search can use

#define grid search
grid = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3
)
grid_result = grid.fit(X_train_final, y_train) #run search

#print best parameters
print("Best Hyperparameters:", grid_result.best_params_)
print("Best Training Accuracy:", grid_result.best_score_)

#extract best hyperparameters
best_fc = grid_result.best_params_["model__fc_units"]
best_dropout = grid_result.best_params_["model__use_dropout"]


#rebuild model using best parameters
best_model = build_model(fc_units=best_fc, use_dropout=best_dropout)
#train model
best_model.fit(
    X_train_final, y_train,
    epochs=50,
    batch_size=32,
    verbose=1
)
#evaluate on test set
test_loss, test_acc = best_model.evaluate(X_test_final, y_test)
print("Test Accuracy with Best Hyperparameters:", test_acc)

c:\Users\sixel\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\sixel\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\sixel\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


c:\Users\sixel\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\sixel\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\sixel\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\sixel\A

Best Hyperparameters: {'batch_size': 32, 'epochs': 50, 'model__fc_units': 2, 'model__use_dropout': True}
Best Training Accuracy: 0.9233716475095785
Epoch 1/50


c:\Users\sixel\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9087 - loss: 0.4295  
Epoch 2/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9083 - loss: 0.3940 
Epoch 3/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9278 - loss: 0.2886 
Epoch 4/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9055 - loss: 0.2935 
Epoch 5/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9429 - loss: 0.2475 
Epoch 6/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9281 - loss: 0.2794 
Epoch 7/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9148 - loss: 0.2671 
Epoch 8/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9230 - loss: 0.2709 
Epoch 9/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1000us/step - accuracy: 0.9217 - loss: 0.2258
Epoch 10/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9199 - loss: 0.2378 
Epoch 11/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9379 - loss: 0.2393 
Epoch 12/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9179 - loss: 0.2480 
Epoch 13/

### Question 6 - Discussion (5%)
- Q6.1 Indicate other hyperparameters relevant to transformers that can be tuned.
- Q6.2 What are the advantages and disadvantages of using GridSearch for finding optimal hyperparameters?


*Write your Answer to Q6.1 and Q6.2 Here:*

Q6.1: Other hyperparameters can be tunable, in terms of model architecture, common ones include number of attention heads (controls how many parallel attention mechanisms the model uses), number of encoder/decoder layers (affects model depth and representation power), hidden dimension size (size of token embeddings and internal representations), feed forward size (width of MLP inside each transformer block). 
For attention and regularization parameters include dropout rate, attention drop out, and layer normalization type.
Lastly, training hyperparameters can include; learning rate, batch size and weight decay. 

Q6.2: Advantages include;
- exhaustive and systematic; tests every combination so it will find the best option within the defined grid.
- simple to implement and interpret; easy to use and straightforward to analyze.
- reproducible; because it tries all predefined values, results are deterministic. 
Disadvantages include;
- extremely computationally expensive; training many models is slow.
- scales poorly; adding even a small amount of hyperparameter values causes the grid size to explode.
- not flexible; only tests the specific values you choose. 
- inefficient for large and continuous hyperparameter spaces.
Overall, GridSearch is reliable for small, controlled search spaces but becomes impractical and inefficient as model complexity or parameter ranges grow. 